In [7]:
import tensorflow  as tf
from tensorflow import keras
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np
import os
import matplotlib.pyplot as plt
from PIL import Image

In [1]:
# Cell 1: Setup Kaggle (YOU KNOW THIS!)
!pip install -q kaggle
import os
import json

# YOUR credentials
os.environ['KAGGLE_USERNAME'] = "tajwerfatima"  # REPLACE!
os.environ['KAGGLE_KEY'] = "KGAT_a8a1c3955b19b6f4831d741db0efbf9f"          # REPLACE!

!mkdir -p ~/.kaggle

kaggle_creds = {
    "username": os.environ['KAGGLE_USERNAME'],
    "key": os.environ['KAGGLE_KEY']
}

with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_creds, f)

!chmod 600 /root/.kaggle/kaggle.json

print("✅ Kaggle ready!")

✅ Kaggle ready!


In [2]:
# Cell 2: Download Vehicle Type Recognition Dataset
print("🚗 Downloading Vehicle Type Recognition dataset...")
!kaggle datasets download -d kaggleashwin/vehicle-type-recognition
!unzip -q vehicle-type-recognition.zip -d /content/vehicles_data

print("✅ Dataset downloaded!")

🚗 Downloading Vehicle Type Recognition dataset...
Dataset URL: https://www.kaggle.com/datasets/kaggleashwin/vehicle-type-recognition
License(s): apache-2.0
 86% 136M/159M [00:00<00:00, 1.42GB/s]
100% 159M/159M [00:00<00:00, 1.30GB/s]
✅ Dataset downloaded!


In [3]:
# Cell 3: Explore What We Got!
import os

data_dir = '/content/vehicles_data'

print("📂 EVERYTHING in vehicles_data:")
for root, dirs, files in os.walk(data_dir):
    level = root.replace(data_dir, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}📁 {os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)

    # Show folders
    if dirs and level < 2:
        print(f'{subindent}Contains folders: {dirs}')

    # Show sample files
    for file in files[:3]:
        print(f'{subindent}📄 {file}')
    if len(files) > 3:
        print(f'{subindent}... and {len(files)-3} more files')

📂 EVERYTHING in vehicles_data:
📁 vehicles_data/
  Contains folders: ['Dataset']
  📁 Dataset/
    Contains folders: ['Bus', 'Truck', 'motorcycle', 'Car']
    📁 Bus/
      📄 Image_19.jpg
      📄 Image_50.jpg
      📄 Image_36.jpg
      ... and 97 more files
    📁 Truck/
      📄 Image_19.jpg
      📄 Image_50.jpg
      📄 Image_36.jpg
      ... and 97 more files
    📁 motorcycle/
      📄 Image_19.jpg
      📄 Image_50.jpg
      📄 Image_36.jpg
      ... and 97 more files
    📁 Car/
      📄 Image_50.jpg
      📄 Image_36.jpg
      📄 Image_16.jpeg
      ... and 97 more files


In [11]:
# Cell 4: Data Generators
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Path to images
data_path = '/content/vehicles_data/Dataset'

# Training generator WITH augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2  # 80/20 split
)

# Validation generator WITHOUT augmentation
val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

# Create generators
train_generator = train_datagen.flow_from_directory(
    data_path,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

validation_generator = val_datagen.flow_from_directory(
    data_path,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation',
    shuffle = False
)

print(f"\n✅ Found {train_generator.samples} training images")
print(f"✅ Found {validation_generator.samples} validation images")
print(f"✅ Categories: {list(train_generator.class_indices.keys())}")

Found 320 images belonging to 4 classes.
Found 80 images belonging to 4 classes.

✅ Found 320 training images
✅ Found 80 validation images
✅ Categories: ['Bus', 'Car', 'Truck', 'motorcycle']


In [5]:
# Cell 5: Build Model
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

# Load pre-trained MobileNetV2
base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)

# Freeze base model
base_model.trainable = False

# Build classifier
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(4, activation='softmax')  # 4 classes!
])

# Compile
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("✅ Model built!")
model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
✅ Model built!


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 4)              │         5,124 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,263,108 (8.63 MB)

 Trainable params: 5,124 (20.02 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [6]:
# Cell 6: Train!
print("🚗 Training vehicles classifier...\n")

history = model.fit(
    train_generator,
    epochs=10,
    validation_data=validation_generator
)

print("\n✅ Training complete!")

🚗 Training vehicles classifier...



/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch 1/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 36s 3s/step - accuracy: 0.2850 - loss: 1.6781 - val_accuracy: 0.8000 - val_loss: 0.6849
Epoch 2/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 37s 3s/step - accuracy: 0.6842 - loss: 0.8657 - val_accuracy: 0.9125 - val_loss: 0.3469
Epoch 3/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 28s 3s/step - accuracy: 0.8401 - loss: 0.4564 - val_accuracy: 0.9500 - val_loss: 0.2311
Epoch 4/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 40s 3s/step - accuracy: 0.8798 - loss: 0.3803 - val_accuracy: 0.9625 - val_loss: 0.1461
Epoch 5/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 27s 3s/step - accuracy: 0.8778 - loss: 0.3269 - val_accuracy: 0.9625 - val_loss: 0.1528
Epoch 6/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 27s 3s/step - accuracy: 0.9197 - loss: 0.2706 - val_accuracy: 0.9625 - val_loss: 0.1199
Epoch 7/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 27s 3s/step - accuracy: 0.9119 - loss: 0.2702 - val_accuracy: 0.9500 - val_loss: 0.1202
Epoch 8/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 27s 3s/step - accuracy: 0.9281 - loss: 0.2093 - val_accuracy: 0.9500 - val_loss:

In [12]:
# Cell 7: FIXED Per-Class Metrics
from sklearn.metrics import classification_report
import numpy as np

# RESET THE GENERATOR FIRST!!! (This was missing!)
validation_generator.reset()

# Now predict
val_predictions = model.predict(validation_generator)
val_pred_classes = np.argmax(val_predictions, axis=1)

# Get true labels AFTER reset
val_true_classes = validation_generator.classes

category_names = list(validation_generator.class_indices.keys())

print("="*70)
print("PER-CLASS PERFORMANCE ANALYSIS")
print("="*70)
print(classification_report(val_true_classes, val_pred_classes,
                           target_names=category_names))

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


3/3 ━━━━━━━━━━━━━━━━━━━━ 5s 958ms/step
PER-CLASS PERFORMANCE ANALYSIS
              precision    recall  f1-score   support

         Bus       0.83      1.00      0.91        20
         Car       1.00      0.95      0.97        20
       Truck       1.00      0.85      0.92        20
  motorcycle       1.00      1.00      1.00        20

    accuracy                           0.95        80
   macro avg       0.96      0.95      0.95        80
weighted avg       0.96      0.95      0.95        80

